# HIT-ASR — main results on People's Speech

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyin-karaca/hit-asr/blob/main/notebooks/main_peoples_speech.ipynb)

This notebook reproduces the main experiment of the paper on People's Speech. Every model's hyperparameters are selected on a
25 % hold-out of the corpus; every model is then trained and tested on the same ten folds of a 5×2 cross-validation
of the remaining clips, and HIT-ASR is compared with every other system on those folds.

The configuration is fixed in `hitasr.configs.MAIN["peoples_speech"]` and the code is in `hitasr.experiment`. The data and the
stored searches and folds are read from the Hugging Face dataset [`huseyin-karaca/hit-asr`](https://huggingface.co/datasets/huseyin-karaca/hit-asr); nothing is written to it.

| Level | What runs | Needs |
|---|---|---|
| **1** | every table, from the searches and folds stored on the Hub | a CPU, minutes |
| **2** | every fold retrained from the published labels and frames (`RETUNE = True`: the search as well) | a GPU, hours |
| **3** | level 2 on labels and frames rebuilt from the audio: run `extract` first on the same machine, which writes them to a local folder that level 3 reads (`HITASR_HUB` names another place) | a GPU |

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("hitasr") is None:     # Colab: install the package (a local or Docker run already has it)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "hit-asr[paper] @ git+https://github.com/huseyin-karaca/hit-asr.git@main"], check=True)

In [2]:
from hitasr.configs import MAIN
from hitasr.experiment import MainExperiment

LEVEL = 1          # 1, 2 or 3 (see above)
RETUNE = False     # levels 2 and 3: run the hyperparameter search again
DEVICE = "cuda"    # levels 2 and 3 train on it

exp = MainExperiment(MAIN["peoples_speech"], level=LEVEL, retune=RETUNE, device=DEVICE)

People's Speech: level 1 — search read from the Hub, folds read from the Hub
experts  Granite-Speech-4.1-2b, Parakeet-CTC-1.1b, Parakeet-TDT-0.6b-v3
arms     HIT-ASR, MLP-pool, ADASTT, ROVER (confidence-weighted), CN-MBR, MLP-pool (hard CE)


## 1. Hyperparameter spaces (Table 2)

The free hyperparameters of every model and their choices; every other parameter is fixed
(`hitasr.configs.SPACES`).

In [3]:
exp.spaces()

Table 2 — hyperparameter spaces
10 random draws per arm; parameters not listed are fixed (hitasr.configs.SPACES)
  [HIT-ASR  (16 free of 20)]
      parameter                                     choices
        d_model                                  {128, 256}
        n_heads                                      {4, 8}
  stage1_layers                                   {1, 2, 3}
  stage2_layers                                      {1, 2}
        ffn_dim                            {256, 512, 1024}
        dropout                                {0.05, 0.15}
         fusion           {bridge, self_attn, concat, mean}
   share_stage1                               {True, False}
     input_norm                  {none, scale, standardize}
             lr                                    {0.0001}
   weight_decay                          {0.001, 0.01, 0.1}
     batch_size                             {8, 16, 32, 64}
     max_frames                           {500, 1000, 2000}
       patience                                     {5, 10}
    loss_preset {hard_ce_only, soft_ce_only, wer+hard, all}
label_smoothing                                  {0.0, 0.1}
  [MLP-pool  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}
  [ADASTT  (7 free of 8)]
       parameter                choices
    n_estimators       {100, 400, 1000}
   learning_rate {0.01, 0.03, 0.1, 0.3}
       max_depth              {3, 5, 8}
       subsample        {0.6, 0.8, 1.0}
colsample_bytree   {0.1, 0.3, 0.6, 1.0}
min_child_weight       {1.0, 4.0, 16.0}
      reg_lambda       {0.3, 3.0, 30.0}
  [ROVER (confidence-weighted)  (6 free of 6)]
     parameter                          choices
         alpha                            {0.0}
   conf_source               {cn, probe, blend}
     weighting {uniform, inverse_wer, log_odds}
weight_by_conf                    {False, True}
     conf_temp                            {2.0}
  null_penalty                      {0.25, 0.5}
  [CN-MBR  (7 free of 7)]
  parameter                          choices
     decode                      {slot, hyp}
 candidates              {members+consensus}
      prior {uniform, inverse_wer, log_odds}
temperature                           {10.0}
 null_prior                      {0.05, 0.2}
   use_conf                    {False, True}
conf_source                          {probe}
  [MLP-pool (hard CE)  (8 free of 12)]
      parameter                        choices
       d_hidden         {256, 512, 1024, 2048}
       n_layers                      {1, 2, 3}
        dropout              {0.05, 0.15, 0.3}
             lr {3e-05, 0.0001, 0.0003, 0.001}
   weight_decay             {0.001, 0.01, 0.1}
     batch_size             {32, 64, 128, 256}
       patience                        {5, 10}
label_smoothing                     {0.0, 0.1}

## 2. Data

The three experts' transcripts and word-error counters for every clip (at levels 2 and 3, also their frame-level
encoder states). The stored counters are recomputed from the stored transcripts as a check.

In [4]:
exp.load()

peoples_speech: 14,299 utterances x 3 experts (0 adopted from fastt) in 2.4s
100.0% of 1,200 (utterance, model) counters reproduce exactly


RouterStore(peoples_speech, granite_speech_4_1_2b+parakeet_ctc_1_1b+parakeet_tdt_0_6b_v3, 14,299 rows, frames closed)

## 3. Hyperparameter search

A random 25 % of the clips is held out for the search. For every model, ten configurations are drawn at random from
its space (Table 2), each trained on 70 % of the hold-out and scored on the remaining 30 %; the configuration with the
lowest corpus WER is kept. The other 75 % of the clips are never seen by the search.

In [5]:
exp.search()

partition,n_utts,share
tune,3575,0.250
eval,10724,0.750
fit,2502,0.175
val,1073,0.075


  on the tuning-score rows: bsm(granite_speech_4_1_2b) 0.1997   oracle 0.1300   gap 0.0698
  the 10,724 evaluation rows are never seen by a trial
  HIT-ASR 10 trials; MLP-pool 10 trials; ADASTT 10 trials; ROVER (confidence-weighted) 10 trials; CN-MBR 10 trials; MLP-pool (hard CE) 10 trials — read from the Hub


HPTRun('hpt_peoples_speech_b6982141', 6 arm(s), best hit_asr 0.1581, eval on 10,724 rows)

In [6]:
exp.dataset()

statistic,peoples_speech
Total clips,14299
Hyperparameter hold-out,3575
Evaluation rows (5x2),10724
Train size (per fold),5362
Test size (per fold),5362
Hours,28.2221
Mean clip length (s),7.1054
Reference words,286169


In [7]:
exp.selected()

label,trials,hold-out WER,selected configuration
HIT-ASR,10,0.1581,"{""batch_size"": 32, ""d_model"": 128, ""dropout"": 0.05, ""ffn_dim"": 256, ""fusion"": ""mean"", ""input_norm"": ""standardize"", ""label_smoothing"": 0.1, ""loss_preset"": ""soft_ce_only"", ""lr"": 0.0001, ""max_frames"": 500, ""max_restarts"": 0, ""n_heads"": 4, ""n_seeds"": 3, ""patience"": 10, ""pooled_skip"": false, ""share_stage1"": true, ""stage1_layers"": 2, ""stage2_layers"": 1, ""weight_decay"": 0.1, ""word_weighted"": false}"
MLP-pool,10,0.1624,"{""batch_size"": 64, ""d_hidden"": 2048, ""dropout"": 0.05, ""label_smoothing"": 0.1, ""loss_preset"": ""all"", ""lr"": 3e-05, ""max_restarts"": 0, ""n_layers"": 3, ""n_seeds"": 3, ""patience"": 10, ""weight_decay"": 0.01, ""word_weighted"": false}"
ADASTT,10,0.1702,"{""colsample_bytree"": 0.1, ""learning_rate"": 0.3, ""max_depth"": 8, ""min_child_weight"": 16.0, ""n_estimators"": 400, ""objective"": ""cross_entropy"", ""reg_lambda"": 3.0, ""subsample"": 0.8}"
ROVER (confidence-weighted),10,0.1820,"{""alpha"": 0.0, ""conf_source"": ""blend"", ""conf_temp"": 2.0, ""null_penalty"": 0.5, ""weight_by_conf"": true, ""weighting"": ""uniform""}"
CN-MBR,10,0.1935,"{""candidates"": ""members+consensus"", ""conf_source"": ""probe"", ""decode"": ""slot"", ""null_prior"": 0.2, ""prior"": ""inverse_wer"", ""temperature"": 10.0, ""use_conf"": false}"
MLP-pool (hard CE),10,0.1706,"{""batch_size"": 64, ""d_hidden"": 512, ""dropout"": 0.3, ""label_smoothing"": 0.1, ""loss_preset"": ""hard_ce_only"", ""lr"": 0.0001, ""max_restarts"": 0, ""n_layers"": 1, ""n_seeds"": 3, ""patience"": 5, ""weight_decay"": 0.01, ""word_weighted"": false}"


## 4. 5×2 cross-validation

Five repetitions, each splitting the evaluation clips into two halves with its own seed. Every model is trained on
one half and tested on the other, then the other way round: ten folds, the same for every model.

In [8]:
exp.cross_validate();

  hub: runlog main_peoples_speech — 35 of 35 (seed, group) result(s) available; fetched 70 file(s) in 1s
  runlog HIT  cv5x2 seed 123 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 7 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 256 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 512 — 7/7 group(s) cached, nothing refitted
  runlog HIT  cv5x2 seed 1234 — 7/7 group(s) cached, nothing refitted
5 repetitions x 2 folds x 15 arms


## 5. Main results (Table 3)

In [9]:
exp.results()

Table 3 — main results, People's Speech
mean ± sd over 10 folds; WER and uWER (mean utterance WER) as fractions, GC (share of the gap between the best single expert and the oracle closed) and Acc (selection accuracy) in %; dist = share of clips routed to Granite-Speech-4.1-2b / Parakeet-CTC-1.1b / Parakeet-TDT-0.6b-v3
  [expert]
                                       arm             WER            uWER         GC@0      Acc@0           dist
                     Granite-Speech-4.1-2b 0.1739 ± 0.0026 0.2537 ± 0.0100    0.0 ± 0.0 67.3 ± 0.5 1.00-0.00-0.00
                         Parakeet-CTC-1.1b 0.1839 ± 0.0016 0.2924 ± 0.0039  -21.8 ± 5.5 49.5 ± 0.3 0.00-1.00-0.00
                      Parakeet-TDT-0.6b-v3 0.1877 ± 0.0087 0.2748 ± 0.0058 -29.3 ± 12.8 60.3 ± 0.6 0.00-0.00-1.00
Best single expert (Granite-Speech-4.1-2b) 0.1739 ± 0.0026 0.2537 ± 0.0100    0.0 ± 0.0 67.3 ± 0.5 1.00-0.00-0.00
  [reference]
   arm             WER            uWER         GC@0      Acc@0           dist
Random 0.1832 ± 0.0089 0.2730 ± 0.0118 -19.7 ± 15.6 59.4 ± 0.9 0.33-0.33-0.33
  [fusion]
                        arm             WER            uWER         GC@0 Acc@0 dist
ROVER (confidence-weighted) 0.1815 ± 0.0023 0.2714 ± 0.0062  -16.5 ± 2.8     —    —
                     CN-MBR 0.1912 ± 0.0117 0.3000 ± 0.0121 -36.7 ± 18.7     —    —
  [pooled]
     arm             WER            uWER       GC@0      Acc@0           dist
MLP-pool 0.1629 ± 0.0032 0.2436 ± 0.0103 23.6 ± 3.1 70.5 ± 0.7 0.70-0.15-0.15
  ADASTT 0.1678 ± 0.0023 0.2479 ± 0.0084 13.1 ± 2.0 69.2 ± 0.7 0.92-0.07-0.02
  [ours]
    arm             WER            uWER       GC@0      Acc@0           dist
HIT-ASR 0.1580 ± 0.0019 0.2351 ± 0.0063 34.0 ± 3.6 71.6 ± 0.4 0.45-0.22-0.32
  [oracle]
                      arm             WER            uWER        GC@0       Acc@0           dist
                   Oracle 0.1274 ± 0.0013 0.1830 ± 0.0034 100.0 ± 0.0 100.0 ± 0.0 0.67-0.18-0.14
Oracle (1-word tolerance) 0.1354 ± 0.0013 0.1998 ± 0.0041  82.8 ± 1.1  83.9 ± 0.4 0.83-0.09-0.07
Oracle (2-word tolerance) 0.1431 ± 0.0013 0.2150 ± 0.0042  66.0 ± 2.0  76.1 ± 0.4 0.91-0.05-0.04

In [10]:
exp.routing()

router,Granite-Speech-4.1-2b,Parakeet-CTC-1.1b,Parakeet-TDT-0.6b-v3,entropy,switch rate
Random,33.5,33.1,33.4,1.585,66.5
MLP-pool,69.8,15.1,15.1,1.185,30.2
ADASTT,91.7,6.6,1.7,0.474,8.3
HIT-ASR,45.5,22.1,32.4,1.525,54.5
Oracle,67.3,18.3,14.3,1.235,32.7
Oracle (1-word tolerance),83.4,9.2,7.4,0.813,16.6
Oracle (2-word tolerance),91.3,5.1,3.7,0.514,8.7


## 6. Statistical test (Table 6)

HIT-ASR against every other system on the same ten folds, with a custom statistical test. For full details, please
see the paper.

In [11]:
exp.tests()

Table 6 — HIT-ASR against every system, People's Speech
on the same 10 folds; effect in the metric's own units
anchor: HIT-ASR; positive effect = anchor better
  [corpus WER]
                 comparator  effect       t      p
      Granite-Speech-4.1-2b +0.0158   6.525 0.0013
          Parakeet-CTC-1.1b +0.0259  15.878 0.0000
       Parakeet-TDT-0.6b-v3 +0.0297   3.387 0.0195
         Best single expert +0.0158   6.525 0.0013
                     Random +0.0252   2.329 0.0673
ROVER (confidence-weighted) +0.0235  11.512 0.0001
                     CN-MBR +0.0332   2.846 0.0360
                   MLP-pool +0.0049   2.928 0.0327
                     ADASTT +0.0098   4.228 0.0083
                     Oracle -0.0306 -21.606 0.0000
  Oracle (1-word tolerance) -0.0227 -15.633 0.0000
  Oracle (2-word tolerance) -0.0149  -9.197 0.0003
  [mean utterance WER]
                 comparator  effect       t      p
      Granite-Speech-4.1-2b +0.0186   2.368 0.0641
          Parakeet-CTC-1.1b +0.0573  14.287 0.0000
       Parakeet-TDT-0.6b-v3 +0.0397   4.345 0.0074
         Best single expert +0.0186   2.368 0.0641
                     Random +0.0379   2.844 0.0361
ROVER (confidence-weighted) +0.0363   6.066 0.0018
                     CN-MBR +0.0649   4.769 0.0050
                   MLP-pool +0.0085   1.987 0.1036
                     ADASTT +0.0128   2.956 0.0317
                     Oracle -0.0522 -13.391 0.0000
  Oracle (1-word tolerance) -0.0354 -11.283 0.0001
  Oracle (2-word tolerance) -0.0202  -5.534 0.0026
Custom statistical test — for full details, please see the paper.

## 7. Record

The configuration, the selected hyperparameters and every table above, in `main_peoples_speech.json`.

In [12]:
exp.save();

wrote main_peoples_speech.json
